# 02 — Clean

**billboard-hot100** — clean the two raw ingest tables into the typed building
blocks that `03-prepare` will shape into the genre-rank-over-time chart data.
All cleaning happens in **DuckDB SQL** (no pandas chained transforms).

This stage produces exactly two interim tables — **nothing is joined or
aggregated across them here** (that is `03-prepare`'s job):

1. **`hot100_clean`** — the ~355k chart appearances, typed and deduped, with
   `year`/`decade` derived and `this_week` renamed to `rank`.
2. **`artist_genre`** — one row per enriched artist, mapping the messy raw
   MusicBrainz tags to ONE aggregate genre via a documented precedence ruleset
   (`src/clean_quality.bucket_genre`).

> ✅ **Coverage: full enrichment set pulled + unmatched re-queried.** The
> MusicBrainz cache (`data/raw/mb_artist_cache.jsonl`) covers all ~4,427 artists
> with ≥ 20 Hot 100 appearances. A first pass queried the raw Billboard credit
> with an exact-quoted search and left ~1,281 unmatched (credit formatting like
> "Elvis Presley With The Jordanaires" never matches MB's canonical name); those
> were **re-queried with normalized primary-artist names** (see the
> name-normalization note below). The cleaning logic is INDEPENDENT of coverage:
> an artist with no usable tags still buckets to **Other/Unknown**. The genuine
> long tail of artists with **< 20 appearances** is below the enrichment threshold
> and stays Other/Unknown by design (documented residual).

### Artist-name normalization + re-query of the unmatched (the big coverage win)

**Problem.** MusicBrainz keys on an artist's **canonical** name, but Billboard
credit strings carry featuring / backing-band / collaboration formatting that
doesn't match it — "Elvis Presley With The Jordanaires" (458 appearances),
"Prince And The Revolution" (139), "Drake & 21 Savage" (103), "Paul McCartney
And Wings" (101). A first ingest pass querying the raw credit (exact-quoted)
left **1,281 of 4,427** enrichment artists unmatched, dumping ~11% of all chart
appearances into Other/Unknown purely as a **name-match failure**, not missing
data.

**Fix (`src/ingest.py`).** `normalize_artist_for_mb()` derives the PRIMARY
artist for lookup: cut featuring/with/duet/vs credits, strip orchestra/session-
band and parenthetical tails, and split a true co-bill to the FIRST artist —
while KEEPING Motown-style "<Lead> & The <Group>" names whole ("Diana Ross &
The Supremes", "Martha & The Vandellas"). Taking the primary of a genuine
co-bill ("Drake & 21 Savage" → Drake) is a conscious choice for an ARTIST-level
genre. A small **curated alias map** handles stubborn names cleaning can't fix
(`Daryl Hall John Oates`→`Daryl Hall & John Oates`, `'N Sync`→`*NSYNC`, `Jackson
5`→`The Jackson 5`, `The 4 Seasons`→`The Four Seasons`, `Dixie Chicks`→`The
Chicks`). The unmatched set was then **re-queried** with the normalized name and
a forgiving (unquoted) query via `requery_unmatched_mb()`, appending improved
records to the SAME resumable cache keyed by the original Billboard spelling (so
the join below is unaffected; the normalized `query_name` is recorded for audit).
This cell simply consumes the now-fuller cache.

In [ ]:
import sys, os, json
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
pd.set_option('display.max_colwidth', 80)

from src.ingest import load_config
from src.clean_quality import (
    get_connection, run_sql, quality_report, save_interim, register_source,
    bucket_genre, genre_fields, AGGREGATE_GENRES,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print('hot100_raw          :', con.execute('SELECT COUNT(*) FROM hot100_raw').fetchone()[0], 'rows')
print('mb_artist_genres_raw:', con.execute('SELECT COUNT(*) FROM mb_artist_genres_raw').fetchone()[0], 'rows')
print('  matched           :', con.execute('SELECT COUNT(*) FROM mb_artist_genres_raw WHERE matched').fetchone()[0])

## Step 1 — clean the chart appearances → `hot100_clean`

From `hot100_raw` (one row per weekly chart appearance). In DuckDB SQL we:
- Cast `date` → DATE and derive `year` (INT) and `decade` (INT, the decade's
  first year, `FLOOR(year/10)*10`).
- Trim/collapse whitespace in `song` and `artist`. We KEEP the untouched
  artist string as **`artist_raw`** — the genre join in `03-prepare` keys on the
  exact Billboard spelling, so the join key must survive cleaning verbatim.
- Rename `this_week` → **`rank`** (the chart position that week; clearer than the
  source's column name) and keep `last_week`, `peak_position`, `weeks_on_chart`.

No genre, no aggregation — just a typed, trimmed copy of the appearances.

In [ ]:
clean = run_sql('''
    SELECT
        CAST(date AS DATE)                           AS date,
        CAST(YEAR(CAST(date AS DATE)) AS INTEGER)    AS year,
        CAST(FLOOR(YEAR(CAST(date AS DATE)) / 10) * 10 AS INTEGER) AS decade,
        REGEXP_REPLACE(TRIM(song),   '\\s+', ' ', 'g') AS song,
        REGEXP_REPLACE(TRIM(artist), '\\s+', ' ', 'g') AS artist,
        artist                                       AS artist_raw,
        CAST(this_week      AS INTEGER)              AS rank,
        CAST(last_week      AS INTEGER)              AS last_week,
        CAST(peak_position  AS INTEGER)              AS peak_position,
        CAST(weeks_on_chart AS INTEGER)              AS weeks_on_chart
    FROM hot100_raw
''', con)
print('typed rows:', f'{len(clean):,}')
clean.head(5)

### Dedupe exact-duplicate (date, song, artist) appearances
A song can only occupy one position on a given weekly chart, so any exact
`(date, song, artist)` duplicate is a source artifact. We report how many we drop.

In [ ]:
before = len(clean)
clean = clean.drop_duplicates(subset=['date', 'song', 'artist']).reset_index(drop=True)
dropped = before - len(clean)
print(f'dropped {dropped:,} exact (date, song, artist) duplicate(s)')
print(f'rows now: {len(clean):,}')

### QC — assert the invariants, then a quality report
`rank` must be 1–100; `date`, `song`, `artist` must be non-null. The
`quality_report()` helper prints null counts, row count and duplicate count.

In [ ]:
# Hard invariants (fail loudly if the source drifts)
assert clean['rank'].between(1, 100).all(), 'rank outside 1..100'
assert clean['date'].notna().all(),   'null date'
assert clean['song'].notna().all(),   'null song'
assert clean['artist'].notna().all(), 'null artist'
assert (clean['artist'].str.len() > 0).all(), 'empty artist string'
print('✓ invariants hold: rank 1-100, no null/empty date/song/artist')

print(f'\ndate range     : {clean["date"].min()} → {clean["date"].max()}')
print(f'distinct artists: {clean["artist"].nunique():,}')
print(f'distinct songs  : {clean["song"].nunique():,}')
print(f'years           : {int(clean["year"].min())} → {int(clean["year"].max())}')
print(f'decades         : {sorted(clean["decade"].unique().tolist())}')

quality_report(clean, 'hot100_clean', con,
               required_columns=['date', 'year', 'decade', 'song', 'artist', 'rank'])

### Save `hot100_clean` (DuckDB table + interim Parquet)

In [ ]:
con.execute('DROP TABLE IF EXISTS hot100_clean')
con.execute('CREATE TABLE hot100_clean AS SELECT * FROM clean')
save_interim(clean, cfg, 'hot100_clean.parquet')
print('hot100_clean:', con.execute('SELECT COUNT(*) FROM hot100_clean').fetchone()[0], 'rows')

## Step 2 — the genre bucketer → `artist_genre` (THE key deliverable)

Billboard has no genre; `mb_artist_genres_raw` carries messy artist-level
MusicBrainz tags (a JSON-string `tags` column of `{name, count}` objects). We
collapse that tag soup to ONE **aggregate genre** per artist with a deterministic,
documented ruleset in `src/clean_quality.py` — the mapping is the real deliverable,
and it does NOT depend on which artists happen to be cached yet.

**Canonical aggregate-genre set** (the only values the bucketer can return):
> Pop, Rock, Hip-Hop/Rap, R&B/Soul, Country, Latin, Dance/Electronic, Folk, Jazz, Blues, Metal, **Other/Unknown**


### 2a. Normalize tags & drop noise (what the bucketer does first)

Before any genre rule fires, each raw tag is **normalized** and noise is
**discarded** (`_normalize_tag` + `_is_noise` in `src/clean_quality.py`):

- **Normalize** (one concept → one token): lowercase/trim; unify
  `r&b` / `rnb` / `rhythm & blues` / `rhythm and blues` → `r&b`;
  `synthpop` / `synth pop` → `synth-pop`; `hip-hop` → `hip hop`;
  `rock n roll` / `rock & roll` → `rock and roll`; slashes/underscores → space.
- **Drop as noise** (never a genre): nationalities/places (`american`, `english`,
  `british`, `uk`, `usa`…), decade/era tags (`80s`, `2010s`, bare `2008`…), and
  role/junk tags (`singer-songwriter`, `female vocals`, `icon`, `ballad`,
  `christmas music`, `2008 universal fire victim`, `classic pop and rock`…).

A couple of worked examples so the behaviour is visible inline:

In [ ]:
from src.clean_quality import _normalize_tag, _is_noise
examples = ['Hip-Hop', 'RnB', 'Rhythm & Blues', 'Synth Pop', 'Rock & Roll',
            'american', '2010s', '2008 universal fire victim', 'female vocals',
            'singer-songwriter', 'pop rock', 'country pop', 'dance-pop']
for e in examples:
    n = _normalize_tag(e)
    print(f'  {e:30s} -> {n:22s} {"[NOISE, dropped]" if _is_noise(n) else ""}')

### 2b. The genre rule — TOP TAG WINS (owner decision 2026-10-14)

An artist's genre is the bucket of its **single highest-MusicBrainz-count
non-noise tag** — NOT a weighted sum across its whole tag profile. This matches
how the public perceives an artist ("when I think Taylor Swift I think Pop",
not Country), rather than crediting an artist's roots/origin genre. It is a
deliberate change from an earlier weighted-sum rule that siphoned crossover
stars into their roots genre (Taylor Swift → Country; Michael Jackson /
Beyoncé → R&B/Soul). Under top-tag-wins those read as **Pop**, because `pop` is
literally their most-voted MB tag.

The bucketer sorts the artist's tags by MB `count` (highest first), skips noise,
and returns the bucket of the FIRST tag that maps to one. Each tag is mapped by
substring keyword rules walked in a fixed **precedence order**, which is used
only as the **tiebreaker when two top tags have the exact same count** (a
specific genre then beats the catch-all `pop`):

> **Metal > Hip-Hop/Rap > Country > Latin > R&B/Soul > Dance/Electronic > Folk
> > Jazz > Blues > Rock > Pop > Other/Unknown**

Rules are **substring/keyword**, not exact-match, so `alternative rock`,
`hard rock`, `classic rock` all resolve to Rock; `nu metal`, `heavy metal`,
`metalcore` all resolve to Metal. Full logic + docstring: `bucket_genre()` in
`src/clean_quality.py`.

**Documented bucket-folding decisions:**
- **Disco → Dance/Electronic** (a dance-floor genre) and **Funk → R&B/Soul**
  (a soul/R&B lineage) — rather than their own buckets.
- **Latin** (reggaeton / Latin pop / salsa / bachata / regional Mexican / …) and
  **Metal** (heavy/thrash/nu/metalcore/…) are their OWN aggregate buckets
  (owner decision 2026-10-14). Latin recovers recent-era Latin chart presence
  previously dumped in Other; Metal is broken OUT of Rock (it used to be a Rock
  keyword and got absorbed) so metal acts read as Metal. **Reggae** still lacks
  the tag volume for its own bucket here and falls to **Other/Unknown**.
- Artists with no usable tags (not matched / not cached / only-noise tags)
  → **Other/Unknown**.

**KNOWN CAVEAT (also in SOURCES.md):** crossover superstars whose top MB tag is
`pop` (Michael Jackson, Beyoncé, Taylor Swift, Rihanna, Bruno Mars, Adele) read
as Pop, so Pop runs larger and the roots genres smaller than a lineage-based
taxonomy would. This is intended under "what the public perceives".

### 2c. Build the `artist_genre` table

One row per artist in the enrichment set, carrying the aggregate `genre` plus
the raw tags (`top_tags`) and `n_tags` for auditability. The heavy transform is
the per-artist bucketing (`genre_fields` → `bucket_genre`); we materialize those
three derived fields in pandas, then write the table through DuckDB. We KEEP the
Billboard-spelling `artist` as the join key `03-prepare` will use.

In [ ]:
mb = con.execute('''
    SELECT artist, mb_name, total_appearances, matched, tags
    FROM mb_artist_genres_raw
''').df()

# Derive genre / n_tags / top_tags from the raw tag JSON (the bucketer).
derived = mb['tags'].apply(genre_fields)
mb['genre']    = derived.apply(lambda t: t[0])
mb['n_tags']   = derived.apply(lambda t: t[1])
mb['top_tags'] = derived.apply(lambda t: t[2])

artist_genre = mb[['artist', 'mb_name', 'genre', 'total_appearances',
                   'matched', 'n_tags', 'top_tags']].copy()

# Write through DuckDB (SELECT * FROM the pandas frame) + interim Parquet.
con.execute('DROP TABLE IF EXISTS artist_genre')
con.execute('CREATE TABLE artist_genre AS SELECT * FROM artist_genre')
save_interim(artist_genre, cfg, 'artist_genre.parquet')
print('artist_genre:', con.execute('SELECT COUNT(*) FROM artist_genre').fetchone()[0], 'rows')
artist_genre.head(5)

## Step 3 — audit the bucketing (sanity-check the precedence calls)

Three audits, all inline: the genre distribution, the Other/Unknown share split
into *partial-pull* vs *genuinely-unmatched/untagged*, and a spot-check of
well-known artists against the bucket they landed in.

### 3a. Genre distribution (artists per aggregate bucket)

In [ ]:
dist = run_sql('''
    SELECT genre, COUNT(*) AS n_artists,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM artist_genre
    GROUP BY genre
    ORDER BY n_artists DESC
''', con)
display(dist)

### 3b. Other/Unknown — how much is partial coverage vs genuinely unmatched?

Being honest about coverage: an artist is Other/Unknown either because the MB
pull hasn't reached them yet (`matched=False` — will resolve as the cache fills)
or because MB matched them but they carry no usable genre tags (`matched=True` but
no rule fired). Only the first number shrinks when the pull completes.

In [ ]:
audit = run_sql('''
    SELECT
        COUNT(*) FILTER (WHERE genre = 'Other/Unknown')                       AS other_total,
        COUNT(*) FILTER (WHERE genre = 'Other/Unknown' AND NOT matched)        AS other_not_yet_pulled,
        COUNT(*) FILTER (WHERE genre = 'Other/Unknown' AND matched)            AS other_matched_no_genre,
        COUNT(*)                                                               AS total_artists,
        COUNT(*) FILTER (WHERE matched)                                        AS matched_artists
    FROM artist_genre
''', con).iloc[0]
n = int(audit['total_artists'])
print(f'Other/Unknown total        : {audit["other_total"]:>5,}  ({audit["other_total"]/n:.1%} of all artists)')
print(f'  ├─ not yet pulled (cache) : {audit["other_not_yet_pulled"]:>5,}  (will resolve as the MB pull fills)')
print(f'  └─ matched, no genre tag  : {audit["other_matched_no_genre"]:>5,}  (genuinely unbucketable)')
print(f'matched artists so far     : {audit["matched_artists"]:>5,} / {n:,}')
matched_other_pct = audit['other_matched_no_genre'] / max(int(audit['matched_artists']), 1)
print(f'\nAmong MATCHED artists, Other/Unknown is {matched_other_pct:.1%} — this is the share '
      'that will NOT shrink when the pull completes.')

### 3c. 15-artist spot check — do the precedence calls look right?

The owner's sanity-check list. For each, the aggregate bucket plus the raw tags
that drove it (highest MB count first). Artists not yet cached show as
Other/Unknown with empty tags — a coverage artifact, not a mapping error.

In [ ]:
spot_names = [
    # perception spot-checks (top-tag-wins should read these as Pop)
    'Taylor Swift', 'Michael Jackson', 'Beyonce',
    # recovered via name-normalization / alias / re-query (were Other before)
    'Elvis Presley With The Jordanaires', 'Daryl Hall John Oates',
    'Prince And The Revolution', 'Diana Ross & The Supremes', 'Jackson 5',
    'Martha & The Vandellas', 'Drake & 21 Savage', 'The Weeknd & Ariana Grande',
    'Paul McCartney And Wings',
    # metal bucket (owner wants these in Metal, not Rock) + a Rock control
    'Metallica', 'Linkin Park', 'Nickelback',
    # other sanity checks
    'Johnny Cash', 'Eminem', 'Kendrick Lamar', 'Daft Punk']
lookup = {r['artist']: r for _, r in
          con.execute('SELECT artist, genre, matched, top_tags FROM artist_genre').df().iterrows()}
rows = []
for name in spot_names:
    r = lookup.get(name)
    if r is None:
        rows.append({'artist': name, 'genre': '(not in enrichment set)',
                     'matched': '', 'top_tags': ''})
    else:
        tt = r['top_tags'] or ''
        rows.append({'artist': name, 'genre': r['genre'],
                     'matched': bool(r['matched']),
                     'top_tags': (tt[:70] + '…') if len(tt) > 70 else tt})
display(pd.DataFrame(rows, columns=['artist', 'genre', 'matched', 'top_tags']))

### 3d. Genre distribution by **chart-appearances** — the number that matters

The lead chart weights genres by actual Hot 100 presence, so the honest coverage
metric is Other/Unknown as a share of **chart-appearances**, not of artists. We
join `hot100_clean` to `artist_genre` on the Billboard spelling. Appearances
whose artist is **below the ≥ 20 enrichment threshold** have no `artist_genre`
row at all — we label those `(<20 appearances, not enriched)` and count them
WITH Other/Unknown for the honest total (they are the documented long-tail
residual). We print the overall split and the Other/Unknown share **by decade**.

In [ ]:
appdist = run_sql('''
    SELECT
        COALESCE(ag.genre, '(<20 appearances, not enriched)') AS genre,
        COUNT(*) AS appearances,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM hot100_clean h
    LEFT JOIN artist_genre ag ON h.artist = ag.artist
    GROUP BY 1 ORDER BY appearances DESC
''', con)
display(appdist)

other_share = run_sql('''
    SELECT ROUND(100.0 * COUNT(*) FILTER (
               WHERE ag.genre = 'Other/Unknown' OR ag.genre IS NULL
           ) / COUNT(*), 1) AS other_pct_appearances
    FROM hot100_clean h LEFT JOIN artist_genre ag ON h.artist = ag.artist
''', con).iloc[0]['other_pct_appearances']
print(f'\nOther/Unknown share of ALL chart-appearances '
      f'(incl. the <20 tail): {other_share}%')
print('(was 31.6% before the name-normalization re-query + Latin/Metal buckets)')

In [ ]:
# Other/Unknown share by DECADE (incl. the <20-appearance tail).
by_decade = run_sql('''
    SELECT h.decade,
           COUNT(*) AS appearances,
           ROUND(100.0 * COUNT(*) FILTER (
               WHERE ag.genre = 'Other/Unknown' OR ag.genre IS NULL
           ) / COUNT(*), 1) AS other_pct
    FROM hot100_clean h LEFT JOIN artist_genre ag ON h.artist = ag.artist
    GROUP BY h.decade ORDER BY h.decade
''', con)
display(by_decade)

## Step 4 — provenance

Register the two interim tables in `_sources`, consistent with how `01-ingest`
recorded the raw tables. The methodology note states that genre is an **aggregate
bucketing of artist-level MusicBrainz tags via a documented precedence ruleset**,
and that coverage is **partial** pending the full MB pull (Other/Unknown includes
not-yet-cached artists).

In [ ]:
register_source(
    con, 'hot100_clean',
    'Billboard Hot 100 chart appearances (cleaned)',
    url=cfg['sources']['hot100_all']['url'],
    license=cfg['sources']['hot100_all']['license'],
    notes=('Typed/deduped copy of hot100_raw: date→DATE with year/decade derived, '
           'song/artist whitespace-normalized (artist_raw keeps the exact Billboard '
           'spelling as the genre join key), this_week renamed to rank. One row per '
           '(date, song, artist) chart appearance.'),
    methodology=('02-clean DuckDB SQL pass: cast types, trim/collapse whitespace, '
                 'drop exact (date, song, artist) duplicates, assert rank in 1..100 '
                 'and no null date/song/artist.'),
    series_breaks=('Inherits the hot100_raw methodology break: Hot 100 ranking '
                   'incorporated digital downloads (2005) then streaming (2007+, '
                   'progressively), so chart presence is not measured identically '
                   'across eras.'),
)
register_source(
    con, 'artist_genre',
    'Artist → aggregate genre (bucketed from MusicBrainz tags)',
    url=cfg['sources']['musicbrainz_artist']['url'],
    license=cfg['sources']['musicbrainz_artist']['license'],
    notes=('One aggregate genre per enriched artist, bucketed from the raw '
           'artist-level MusicBrainz tags in mb_artist_genres_raw. artist = the '
           'Billboard spelling (join key); top_tags/n_tags kept for auditability. '
           'Aggregate set: ' + ', '.join(AGGREGATE_GENRES) + '.'),
    methodology=('bucket_genre() in src/clean_quality.py: TOP-TAG-WINS — normalize '
                 'tags (unify r&b/rnb, synth-pop, etc.), drop noise (nationality/'
                 'decade/role tags), sort surviving tags by MB count and take the '
                 'bucket of the single highest-count tag. Each tag maps by substring '
                 'keyword rules; a fixed precedence (Metal > Hip-Hop/Rap > Country > '
                 'Latin > R&B/Soul > Dance/Electronic > Folk > Jazz > Blues > Rock > '
                 'Pop) breaks exact-count ties only. Disco→Dance/Electronic, '
                 'Funk→R&B/Soul; Latin and Metal are their own buckets; Reggae→Other.'),
    series_breaks=('COVERAGE IS PARTIAL pending the full MusicBrainz pull — artists '
                   'not yet cached bucket to Other/Unknown (matched=false). Re-running '
                   'after the pull completes changes only the Other/Unknown share, not '
                   'the mapping. Tags are crowd-sourced and artist-level (whole catalog), '
                   'not per-song or point-in-time.'),
)
from src.clean_quality import get_sources
display(get_sources(con)[['duckdb_table', 'source_name', 'retrieved']])

---
**Next:** `03-prepare.ipynb` — join `hot100_clean` to `artist_genre` on the Billboard artist spelling and build the chart-ready genre-rank-over-time tables (per-year / per-decade genre presence + rank), then export. The join and all aggregation live in 03, NOT here.

---
## Cleanup
Close the DuckDB connection so the single-writer lock is released (the background MB pull writes a JSONL file, not this DB, so it is never the writer).

In [ ]:
con.close()
print('connection closed')